# Machine Learning in Physics: In-class exercises

Each section matches one of the four tasks from the lecture:

1. **Regression**: estimating galaxy redshifts from colors
2. **Classification**: sorting galaxies by shape
3. **Clustering**: finding red and blue galaxies without labels
4. **Dimensionality reduction**: describing galaxy images with a few numbers

**How to use this notebook.** Run the *Setup* cell once, then work on any section you like, in any order. Each section loads its own data and runs on its own. 

**Data**
- SDSS photometry and spectroscopic redshifts (Bhavanam, S. R. (2025). AstroML-Datasets: Introductory Machine Learning Tasks Across Physics and Astronomy [Dataset]. Zenodo. https://doi.org/10.5281/zenodo.17538755)
- Galaxy10 SDSS: 21,785 galaxy images with labels from Galaxy Zoo (Leung, W. H., & Bovy, J. (2019). Galaxy10 SDSS [Dataset]. Zenodo. https://doi.org/10.1093/mnras/sty3217)

**AI Disclaimer:**
This notebook was set-up with the help of Claude (Anthropic).

## Setup

You need `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `h5py` and `requests`. If any are missing, run the following cell:

In [ ]:
#!pip install numpy pandas matplotlib scikit-learn h5py requests

The first time a section runs, it downloads its data (about 150 MB for the redshift catalog and 210 MB for the galaxy images) into a `data/` folder next to this notebook. After that, it uses the local copy.

In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
RNG = np.random.default_rng(42)

FILES = {
    "photoz": {
        "url": "https://zenodo.org/records/17538755/files/PhotoZ_SDSS.csv?download=1",
        "name": "PhotoZ_SDSS.csv",
        "md5": "6d9e6c2e3d027dc609d34e34af42acc1",
    },
    "galaxy10": {
        "url": "https://zenodo.org/records/10844811/files/Galaxy10.h5?download=1",
        "name": "Galaxy10.h5",
        "md5": "3c427aa4d6d985820c7ff1a1608441d8",
    },
}


def md5sum(path, chunk=2**20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def fetch(key):
    """Download a data file the first time it is needed, then reuse the local copy."""
    info = FILES[key]
    path = DATA_DIR / info["name"]
    if path.exists():
        return path
    print(f"Downloading {info['name']} (first run only, this can take a minute)...")
    tmp = path.with_name(path.name + ".part")
    with requests.get(info["url"], stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for block in r.iter_content(chunk_size=2**20):
                f.write(block)
    tmp.rename(path)
    if md5sum(path) != info["md5"]:
        print("Warning: the checksum does not match the published value; the download may be incomplete.")
    return path


# ---------- SDSS photometry + spectroscopic redshifts ----------
BANDS = "ugri"

# The loader finds the magnitude and redshift columns automatically.
# If that fails, write the column names here, e.g.
# MAG_COLUMNS = {"u": "dered_u", "g": "dered_g", "r": "dered_r", "i": "dered_i", "z": "dered_z"}
# REDSHIFT_COLUMN = "specz"
MAG_COLUMNS = None
REDSHIFT_COLUMN = None


def _find(columns, candidates, exclude=()):
    lookup = {c.lower(): c for c in columns if c not in exclude}
    for cand in candidates:
        if cand.lower() in lookup:
            return lookup[cand.lower()]
    return None


def load_photoz(n_max=100_000):
    """Return a table with u, g, r, i magnitudes and the spectroscopic redshift."""
    df = pd.read_csv(fetch("photoz"))
    zcol = REDSHIFT_COLUMN or _find(df.columns, ["specz", "spec_z", "z_spec", "zspec", "redshift", "z"])
    mags = MAG_COLUMNS or {
        b: _find(df.columns,
                 [f"dered_{b}", f"modelMag_{b}", f"cModelMag_{b}", f"petroMag_{b}", f"mag_{b}", f"{b}mag", b],
                 exclude=(zcol,))
        for b in BANDS
    }
    if zcol is None or None in mags.values() or df[zcol].median() > 5:
        raise ValueError(
            "Could not identify the magnitude and redshift columns automatically.\n"
            f"The columns in this file are: {list(df.columns)}\n"
            "Set MAG_COLUMNS and REDSHIFT_COLUMN in the Setup cell and run it again."
        )
    out = pd.DataFrame({b: pd.to_numeric(df[mags[b]], errors="coerce") for b in BANDS})
    out["redshift"] = pd.to_numeric(df[zcol], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    good = ((out[list(BANDS)] > 10) & (out[list(BANDS)] < 30)).all(axis=1) & (out["redshift"] > 0)
    out = out[good]
    if len(out) > n_max:
        out = out.sample(n_max, random_state=42)
    return out.reset_index(drop=True)


# ---------- Galaxy10 SDSS images ----------
# Class names as listed in the Galaxy10 SDSS documentation
GALAXY10_CLASSES = [
    "Disk, face-on, no spiral",
    "Smooth, completely round",
    "Smooth, in-between round",
    "Smooth, cigar-shaped",
    "Disk, edge-on, rounded bulge",
    "Disk, edge-on, boxy bulge",
    "Disk, edge-on, no bulge",
    "Disk, face-on, tight spiral",
    "Disk, face-on, medium spiral",
    "Disk, face-on, loose spiral",
]


def load_galaxy10():
    """Return the images (N, 69, 69, 3; values 0-255) and their class labels (0-9)."""
    import h5py
    with h5py.File(fetch("galaxy10"), "r") as f:
        images = np.array(f["images"])
        labels = np.array(f["ans"]).astype(int)
    return images, labels


def to_gray(images, block=1):
    """Average the g, r, i channels, scale to 0-1, and optionally shrink by averaging block x block pixels."""
    gray = images.mean(axis=-1, dtype=np.float32) / 255.0
    if block > 1:
        n, h, w = gray.shape
        h2, w2 = h // block, w // block
        gray = gray[:, :h2 * block, :w2 * block].reshape(n, h2, block, w2, block).mean(axis=(2, 4))
    return gray


print("Setup done.")

---
## 1. Regression: photometric redshifts

The most reliable way to measure a galaxy's redshift is from its **spectrum**: emission and absorption lines are shifted by a precise, measurable amount. But spectra are expensive. Each one takes far more telescope time than an image, so surveys like SDSS have spectra for only a small fraction of the galaxies they image.

Images, on the other hand, give us the brightness of every galaxy in five color filters (u, g, r, i, z). As a galaxy is redshifted, features in its spectrum (such as the sharp drop in light just blueward of 4000 angstrom) move from one filter into the next, so its colors change (roughly) with redshift.

So the question is: using the galaxies that do have spectra as a training set, can we learn to predict redshift from colors alone?

**Data:** u, g, r, i magnitudes (inputs) and the spectroscopic redshift (labels) for SDSS galaxies.

In [ ]:
# Load data and compute colors (differences between magnitudes in different bands)
gals = load_photoz()

colors = pd.DataFrame({
    "u-g": gals["u"] - gals["g"],
    "g-r": gals["g"] - gals["r"],
    "r-i": gals["r"] - gals["i"],
})
X = colors.to_numpy()
y = gals["redshift"].to_numpy()
print(f"{len(gals):,} galaxies, redshifts from {y.min():.2f} to {y.max():.2f}")

# How does each color change with redshift?
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)
for ax, col in zip(axes, colors.columns):
    ax.hexbin(y, colors[col], gridsize=60, bins="log", mincnt=1, cmap="viridis")
    ax.set_ylim(np.percentile(colors[col], [0.5, 99.5]))
    ax.set_xlabel("spectroscopic redshift")
    ax.set_title(col)
axes[0].set_ylabel("color (mag)")
plt.tight_layout()
plt.show()

Now we train three models of increasing flexibility and compare them. A quarter of the galaxies are held back as a **test set** that the models never see during training.

We measure the error as Δz/(1+z), the standard way to compare photometric redshifts across redshift, and count an **outlier** as any galaxy with |Δz|/(1+z) > 0.15.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

models = {
    "Linear regression": LinearRegression(),
    "k-nearest neighbors": KNeighborsRegressor(n_neighbors=100),
    "Random forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0),
}


def photoz_scores(z_true, z_pred):
    dz = (z_pred - z_true) / (1 + z_true)
    return np.sqrt(np.mean(dz**2)), np.mean(np.abs(dz) > 0.15)


predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    rms, outliers = photoz_scores(y_test, predictions[name])
    print(f"{name:20s}  RMS of dz/(1+z) = {rms:.3f}   outlier fraction = {outliers:.1%}")

# Predicted vs true redshift for each model
zmax = np.percentile(y_test, 99.5)
fig, axes = plt.subplots(1, len(predictions), figsize=(5 * len(predictions), 4.5), sharex=True, sharey=True)
for ax, (name, z_pred) in zip(axes, predictions.items()):
    ax.hexbin(y_test, z_pred, gridsize=60, bins="log", mincnt=1, extent=(0, zmax, 0, zmax), cmap="viridis")
    ax.plot([0, zmax], [0, zmax], "k--", lw=1)
    ax.set_title(name)
    ax.set_xlabel("true (spectroscopic) redshift")
axes[0].set_ylabel("predicted redshift")
plt.tight_layout()
plt.show()

### Questions to think about
- Which model does best? Looking back at the color–redshift plots, why might a straight-line model struggle?
- Where do the predictions go wrong? Look at the highest redshifts. How many training galaxies are out there? Why does it matter?
- Each model gives a single number per galaxy. If you needed an uncertainty on that number, how could you get one?
- What are the biases in the data? For example, the training set only contains galaxies bright enough to get a spectrum. What might happen if you apply the model to fainter galaxies?

### Going further
- Add the r-band magnitude as a fifth input. Does it help? Why might brightness carry redshift information?
- Train only on galaxies with z < 0.3 and test on galaxies with z > 0.3. What does the model predict for them?
- Change `n_neighbors` or `min_samples_leaf`. Can you make a model overfit? 

---
## 2. Classification: galaxy morphology

A galaxy's shape says a lot about its history. Smooth, round galaxies are typically old and have mostly stopped forming stars, while disks with spiral arms are usually still forming stars. 

In the **Galaxy Zoo** project, volunteers classified SDSS galaxies by eye. The Galaxy10 dataset keeps the 21,785 galaxies for which majority of volunteers agreed on one of 10 classes. Can a model learn to reproduce those human labels?

**Data:** 69×69 pixel color images (g, r, i bands), each with one of 10 class labels.

In [ ]:
images, labels = load_galaxy10()
print(f"{len(images):,} images of shape {images.shape[1:]}\n")

counts = np.bincount(labels, minlength=len(GALAXY10_CLASSES))
for k, (name, n) in enumerate(zip(GALAXY10_CLASSES, counts)):
    print(f"class {k}: {name:30s} {n:6,d} galaxies")

# One random example from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for k, ax in enumerate(axes.flat):
    members = np.flatnonzero(labels == k)
    if len(members):
        ax.imshow(images[RNG.choice(members)])
    ax.set_title(f"{k}: {GALAXY10_CLASSES[k]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

To keep things simple, we turn each image into a list of numbers: average the three color channels, shrink the image to 23×23 pixels, and flatten it. Each galaxy becomes **529 numbers**, which any classifier can use.

We compare three classifiers against a deliberately silly baseline that always guesses the most common class.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay

X = to_gray(images, block=3).reshape(len(images), -1)
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25, stratify=labels, random_state=0
)

classifiers = {
    "Always guess the most common class": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
    "Small neural network": make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True, max_iter=200, random_state=0),
    ),
}

predicted = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    predicted[name] = clf.predict(X_test)
    print(f"{name:36s} accuracy = {accuracy_score(y_test, predicted[name]):.2f}   "
          f"balanced accuracy = {balanced_accuracy_score(y_test, predicted[name]):.2f}")

# Where does the random forest go wrong?
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, predicted["Random forest"], labels=range(len(GALAXY10_CLASSES)),
    normalize="true", values_format=".2f", cmap="Blues", colorbar=False, ax=ax,
)
ax.set_title("Random forest confusion matrix (normalized)")
plt.tight_layout()
plt.show()

### Questions to think about
- Compare every model with "always guess the most common class". Is plain accuracy a good score for this dataset? What does balanced accuracy measure?
- Which classes get confused with each other in the confusion matrix? Are those confusions physically reasonable?
- The labels are volunteer votes, not ground truth. What does that mean for the best accuracy you could hope for?
- We flattened each image into a list of pixels. What information about the galaxy did that throw away? (This is what convolutional neural networks are designed to keep.)
- Class 5 has very few examples. What happens to it?

### Going further
- Plot the confusion matrix for the other models and compare the results. What aspects do they perform better or worse in compared to RF?
- A galaxy's class shouldn't depend on how it's rotated. Add rotated and flipped copies of the training images (`np.rot90`, `np.flip`) and retrain. Does it help?
    - A CNN would be designed keeping properties such as translation and rotational symmetry in mind. Where the object appears and its orientation in the image are unrelated to its properties.

---
## 3. Clustering: red and blue galaxies

Nearby galaxies come in two broad populations: a **red sequence** of mostly old galaxies that have largely stopped forming stars, and a **blue cloud** of galaxies that are still forming stars. In SDSS this shows up as two peaks in u−r color, and Strateva et al. (2001) proposed a dividing line at u−r ≈ 2.22.

Here we don't tell the algorithm any of that. Can it find the two populations on its own?

**Data:** the same SDSS photometry as Section 1, restricted to a narrow slice of low redshift so that redshift itself doesn't change the colors much. Within the slice, apparent r magnitude is a rough stand-in for how luminous a galaxy is.

In [ ]:
gals = load_photoz()

ZMIN, ZMAX = 0.02, 0.10
nearby = gals[(gals["redshift"] > ZMIN) & (gals["redshift"] < ZMAX)]
features = pd.DataFrame({"u-r": nearby["u"] - nearby["r"], "r": nearby["r"]})
features = features[(features["u-r"] > 0) & (features["u-r"] < 4.5)]
print(f"{len(features):,} galaxies with {ZMIN} < z < {ZMAX}")
if len(features) < 1000:
    print("That's not many. Try widening ZMIN and ZMAX.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hexbin(features["r"], features["u-r"], gridsize=60, bins="log", mincnt=1, cmap="viridis")
axes[0].axhline(2.22, color="k", ls="--", lw=1, label="u−r = 2.22 (Strateva et al. 2001)")
axes[0].set_xlabel("r magnitude")
axes[0].set_ylabel("u − r color")
axes[0].legend(loc="lower left")
axes[1].hist(features["u-r"], bins=80, color="gray")
axes[1].axvline(2.22, color="k", ls="--", lw=1)
axes[1].set_xlabel("u − r color")
axes[1].set_ylabel("number of galaxies")
plt.tight_layout()
plt.show()

Now we ask two clustering algorithms to split the galaxies into two groups using only color and brightness. The inputs are rescaled first so that both have a comparable range.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

Xc = StandardScaler().fit_transform(features[["u-r", "r"]])

n_clusters = 2
n_components = 2
cluster_models = {
    f"k-means, k = {n_clusters}": KMeans(n_clusters=n_clusters, n_init=10, random_state=0),
    f"Gaussian mixture, {n_components} components": GaussianMixture(n_components=n_components, random_state=0),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (name, model) in zip(axes, cluster_models.items()):
    groups = model.fit_predict(Xc)
    ax.scatter(features["r"], features["u-r"], c=groups, s=2, cmap="coolwarm", alpha=0.4)
    ax.axhline(2.22, color="k", ls="--", lw=1)
    ax.set_title(name)
    ax.set_xlabel("r magnitude")
axes[0].set_ylabel("u − r color")
plt.tight_layout()
plt.show()

### Before we analyze the results, let's see what happens if we try to cluster pure random noise.

In [ ]:
# Clustering always finds groups, even when there are none
# Increase the number of clusters to see what happens when the data is pure noise

n_clusters = 1

noise = RNG.uniform(size=(2000, 2))
noise_groups = KMeans(n_clusters=n_clusters, n_init=10, random_state=0).fit_predict(noise)
plt.figure(figsize=(4.5, 4.5))
plt.scatter(noise[:, 0], noise[:, 1], c=noise_groups, s=5, cmap="tab10")
plt.title("k-means with k = 4 on pure random noise")
plt.tight_layout()
plt.show()

### Questions to think about
- Did either algorithm find the red/blue split? How does its boundary compare with u−r = 2.22?
- k-means draws straight boundaries between groups, while a Gaussian mixture allows elliptical, overlapping groups. Which matches these data better, and why?
- k-means found clusters in pure noise. How would you check whether clusters in real data are physical? 
- We rescaled the inputs before clustering. Try it without `StandardScaler`. What changes, and why?

### Going further
- Try k = 3 or more. Does the extra group have a physical interpretation? (Look up the *green valley*.)
- Add g−r or r−i as extra inputs.
- Try `sklearn.cluster.HDBSCAN`, which decides the number of clusters by itself.

---
## 4. Dimensionality reduction: eigengalaxies

A 69×69 grayscale image is 4,761 numbers. But galaxy images are far from random: most are a bright blob with some size, shape, brightness and orientation, while others are homogenous and spherical lacking any structure.

Principal component analysis (PCA) finds a small set of building-block images, the principal components, so that each galaxy is approximately the average galaxy plus a weighted sum of them. If a handful of weights describes most galaxies well, we've compressed 4,761 numbers into a few numbers.

**Data:** 5,000 randomly chosen Galaxy10 images. The labels are only used at the end, to color one plot.

In [ ]:
from sklearn.decomposition import PCA

images, labels = load_galaxy10()
subset = RNG.choice(len(images), size=min(5000, len(images)), replace=False)
gray = to_gray(images[subset])
sub_labels = labels[subset]
X = gray.reshape(len(gray), -1)
side = gray.shape[1]

pca = PCA(n_components=50, random_state=0).fit(X)
cumulative = np.cumsum(pca.explained_variance_ratio_)
print(f"Each image has {X.shape[1]:,} pixels.")
for n in (1, 5, 10, 20, 50):
    print(f"  {n:2d} components capture {cumulative[n - 1]:.0%} of the variation")

plt.figure(figsize=(5, 3.5))
plt.plot(np.arange(1, 51), cumulative, "o-", ms=3)
plt.xlabel("number of components")
plt.ylabel("fraction of variation captured")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# The average galaxy and the first eight building blocks ("eigengalaxies")
fig, axes = plt.subplots(1, 9, figsize=(18, 2.4))
axes[0].imshow(pca.mean_.reshape(side, side), cmap="gray")
axes[0].set_title("average galaxy")
for k, ax in enumerate(axes[1:]):
    ax.imshow(pca.components_[k].reshape(side, side), cmap="RdBu_r")
    ax.set_title(f"component {k + 1}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Rebuilding a few galaxies from only the first n components
weights = pca.transform(X)
examples = RNG.choice(len(X), size=4, replace=False)
n_list = [1, 5, 20, 50]
fig, axes = plt.subplots(len(examples), len(n_list) + 1, figsize=(2.2 * (len(n_list) + 1), 2.2 * len(examples)))
for row, idx in enumerate(examples):
    axes[row, 0].imshow(gray[idx], cmap="gray")
    axes[row, 0].set_title("original" if row == 0 else "")
    for col, n in enumerate(n_list, start=1):
        rebuilt = pca.mean_ + weights[idx, :n] @ pca.components_[:n]
        axes[row, col].imshow(rebuilt.reshape(side, side), cmap="gray")
        axes[row, col].set_title(f"{n} component{'s' if n > 1 else ''}" if row == 0 else "")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Do smooth and disk galaxies sit in different places?
smooth = np.isin(sub_labels, [1, 2, 3])
plt.figure(figsize=(5.5, 4.5))
plt.scatter(weights[~smooth, 0], weights[~smooth, 1], s=3, alpha=0.4, label="disk classes")
plt.scatter(weights[smooth, 0], weights[smooth, 1], s=3, alpha=0.4, label="smooth classes")
plt.xlabel("weight of component 1")
plt.ylabel("weight of component 2")
plt.legend(markerscale=4)
plt.tight_layout()
plt.show()

### Questions to think about
- How many components capture 90% of the variation? What does that say about how complicated galaxy images really are?
- Look at the first few components. Can you describe what each one changes: size, brightness, elongation, orientation? Remember, the new axes don't have to mean anything physical.
- Do smooth and disk galaxies separate in the plot of the first two weights? If not, what might the first two components be missing?
- The rebuilt images keep the overall shape but lose detail. Which details disappear first?

### Going further
- Use the PCA weights (`pca.transform(...)`) as inputs to the classifiers in Section 2.
- Run t-SNE (`sklearn.manifold.TSNE`) on the PCA weights to make a 2D map, colored by class. Do classes form islands?
- PCA is linear. An **autoencoder**, a neural network, can learn a nonlinear compression. Where might that help?

---
## What's next
Lecture 2 picks up from here: how models are trained (losses and optimization), how to tell whether a model is overfitting, and how to validate it properly. This notebook serves as a starting point for those ideas.